# 05b — Mosaic-aware differential expression: four contrasts

Notebook 05 ran the standard between-genotype comparison across all cell types.
This notebook does the analysis your design actually supports, using the
recombination calls from 03b.

## The four contrasts, and what each one answers

| # | Contrast | Question | Design |
|---|---|---|---|
| **A** | KO-recombined vs WT beta | Full effect of losing Bola3 | `~ genotype`, between animals |
| **B** | KO-recombined vs KO-escaper | **Cell-autonomous effect** | `~ mouse + status`, *paired within animal* |
| **C** | KO-escaper vs WT beta | Systemic / bystander effect | `~ genotype`, between animals |
| **D** | all KO beta vs WT beta | The naive comparison | diluted by mosaicism |

**B is your primary result.** Recombined and escaper cells come from the same
animal: same tamoxifen, same islet isolation, same circulating glucose, same
capture, same sequencing run. Pairing on `mouse_id` removes between-animal
variation, which at n=4 is your dominant noise source. Nothing else in this
analysis is that well controlled.

**C is the contrast that makes B interpretable.** Your KO mice are glucose
intolerant, and chronic hyperglycaemia independently drives beta-cell
dedifferentiation and stress-response activation — the same signature Bola3 loss
predicts. Escaper cells carry the systemic exposure without the lesion. If C is
near-empty, the systemic component is small and A ≈ B. If C is substantial, then
A conflates the two and only B isolates the cell-autonomous effect.

**D is reported for completeness and to quantify what mosaicism cost you.**
Its effect sizes should be ≈ `recombination_rate × B`. Confirming that
relationship holds is a useful check that the classifier worked.

## Read A, B and C together

| A | B | C | Interpretation |
|---|---|---|---|
| large | large | ~empty | Clean cell-autonomous effect. The strongest possible result. |
| large | large | moderate | Both mechanisms operate. Report B as the cell-autonomous component. |
| large | ~empty | large | The change is systemic, not cell-autonomous — driven by the animals' glucose intolerance rather than by Bola3 in that cell. A major reinterpretation. |
| ~empty | large | — | Effect exists but between-animal variance swamped it. Exactly what pairing was for. |

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "scripts"))
sys.path.insert(0, str(Path.cwd().parent / "resources"))

import warnings

import decoupler as dc
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

import config as cfg

warnings.simplefilter("ignore", category=FutureWarning)
sc.settings.set_figure_params(dpi=90, dpi_save=300, frameon=False)

adata = sc.read_h5ad(cfg.H5AD["final"])
adata.X = adata.layers["counts"].copy()
from recombination import attenuation_from_impurity, dilution_factor

rates = pd.read_csv(cfg.TABLE_DIR / "03b_recombination_rates.csv")
MEAN_RATE = rates.recombination_rate.mean()

beta = adata[(adata.obs.cell_type == "beta")
             & (adata.obs.ko_status.isin(["WT", "escaper", "recombined"]))].copy()
print(beta.obs.groupby(["sample_id", "ko_status"], observed=True).size().unstack(fill_value=0))
print(f"\nmean recombination rate: {MEAN_RATE:.1%}")

## 5b.1 Pseudobulk by (sample × status)

Aggregating on `ko_status` rather than `genotype` is the whole point: it gives
each KO mouse **two** pseudobulk profiles, one recombined and one escaper, which
is what makes the paired contrast possible.

In [ ]:
beta.obs["psb_group"] = beta.obs["sample_id"].astype(str) + "__" + beta.obs["ko_status"].astype(str)

pdata = dc.pp.pseudobulk(beta, sample_col="psb_group", groups_col=None,
                         mode="sum", layer="counts")
meta = (beta.obs.drop_duplicates("psb_group")
        .set_index("psb_group")[["sample_id", "mouse_id", "genotype", "ko_status", "batch", "sex"]])
for col in meta.columns:
    pdata.obs[col] = pdata.obs.index.map(meta[col])

NCELLS = "psbulk_cells" if "psbulk_cells" in pdata.obs else "psbulk_n_cells"
keep = ((pdata.obs[NCELLS] >= cfg.MIN_CELLS_PER_PSEUDOBULK)
        & (pdata.obs["psbulk_counts"] >= cfg.MIN_COUNTS_PER_PSEUDOBULK))
print(f"{keep.sum()} / {len(keep)} pseudobulk profiles pass the HNOCA thresholds")
pdata = pdata[keep].copy()
display(pdata.obs[[NCELLS, "psbulk_counts", "genotype", "ko_status"]].sort_index())

# gene exclusions, as in notebook 05
pct_cells = np.asarray((adata.layers["counts"] > 0).mean(axis=0)).ravel() * 100
drop = (set(adata.var_names[pct_cells < 1.0])
        | {"Xist", "Tsix", "Ddx3y", "Eif2s3y", "Uty", "Kdm5d", "Erdr1"}
        | set(adata.var_names[adata.var_names.str.startswith(cfg.MOUSE_MITO_PREFIX)]))
pdata = pdata[:, ~pdata.var_names.isin(drop)].copy()
print(f"{pdata.n_vars} genes retained")

### Do the pseudobulk profiles separate the way the design predicts?

On a PCA, escaper profiles should sit between WT and recombined — or, if the
systemic effect is negligible, on top of WT. Recombined profiles should be the
outliers. Seeing that before any test is reassuring; not seeing it is a signal
to revisit the classification in 03b.

In [ ]:
ppca = pdata.copy()
sc.pp.normalize_total(ppca, target_sum=1e6); sc.pp.log1p(ppca)
sc.pp.pca(ppca, n_comps=min(6, ppca.n_obs - 1))

PALETTE = {"WT": "#4C72B0", "escaper": "#8DA65B", "recombined": "#C44E52"}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
for status, colour in PALETTE.items():
    m = ppca.obs.ko_status == status
    axes[0].scatter(ppca.obsm["X_pca"][m, 0], ppca.obsm["X_pca"][m, 1],
                    c=colour, s=120, label=status, edgecolor="k", linewidth=0.5)
for i, name in enumerate(ppca.obs_names):
    axes[0].annotate(ppca.obs.mouse_id.iloc[i], ppca.obsm["X_pca"][i, :2],
                     fontsize=6, xytext=(5, 3), textcoords="offset points")
axes[0].set_xlabel("PC1"); axes[0].set_ylabel("PC2")
axes[0].legend(frameon=False); axes[0].set_title("pseudobulk PCA")

# within-mouse pairing: does each KO animal move in the same direction?
ko = ppca.obs[ppca.obs.genotype == "KO"]
for mouse in ko.mouse_id.unique():
    pair = ppca.obs.index[(ppca.obs.mouse_id == mouse)]
    if len(pair) == 2:
        idx = [ppca.obs_names.get_loc(p) for p in pair]
        order = np.argsort([ppca.obs.ko_status.iloc[i] != "escaper" for i in idx])
        pts = ppca.obsm["X_pca"][[idx[o] for o in order], :2]
        axes[1].annotate("", xy=pts[1], xytext=pts[0],
                         arrowprops=dict(arrowstyle="->", color="#555", lw=1.5))
for status, colour in PALETTE.items():
    m = ppca.obs.ko_status == status
    axes[1].scatter(ppca.obsm["X_pca"][m, 0], ppca.obsm["X_pca"][m, 1],
                    c=colour, s=110, label=status, edgecolor="k", linewidth=0.5)
axes[1].set_xlabel("PC1"); axes[1].set_ylabel("PC2")
axes[1].set_title("arrows: escaper → recombined, within each KO mouse\n"
                  "parallel arrows = consistent effect", fontsize=9)
sns.despine(fig); fig.tight_layout()
fig.savefig(cfg.FIG_DIR / "05b_pseudobulk_pca.png", dpi=300, bbox_inches="tight")

The right-hand panel is worth dwelling on. **Parallel arrows across all four KO
mice mean the effect is consistent** — the single most persuasive piece of
evidence you can produce at n=4. Arrows pointing in different directions mean
the animals responded differently, and you should find out what distinguishes
them (recombination rate? severity of glucose intolerance?) before pooling.

## 5b.2 Run the four contrasts

In [ ]:
def run_contrast(pdata, subset_mask, design, contrast, label, min_per_group=3):
    sub = pdata[subset_mask].copy()
    factor = contrast[0]
    counts = sub.obs[factor].value_counts()
    if len(counts) < 2 or counts.min() < min_per_group:
        print(f"[{label}] SKIPPED — need >= {min_per_group} per group, have {counts.to_dict()}")
        return None

    genes = dc.pp.filter_by_expr(sub, group=factor, min_count=10,
                                 min_total_count=15, inplace=False)
    sub = sub[:, genes].copy()

    dds = DeseqDataSet(adata=sub, design=design, refit_cooks=True, quiet=True)
    dds.deseq2()
    stats = DeseqStats(dds, contrast=list(contrast), quiet=True)
    stats.summary()

    res = stats.results_df.copy()
    res["gene"] = res.index
    res["contrast"] = label
    n_sig = (res.padj < cfg.FDR_ALPHA).sum()
    n_up = ((res.padj < cfg.FDR_ALPHA) & (res.log2FoldChange > 0)).sum()
    print(f"[{label}] {sub.n_vars} genes tested | {n_sig} at FDR<{cfg.FDR_ALPHA} "
          f"({n_up} up, {n_sig - n_up} down) | design {design}")
    return res


obs = pdata.obs
results = {}

# A -- recombined vs WT, between animals
results["A_recombined_vs_WT"] = run_contrast(
    pdata, obs.ko_status.isin(["recombined", "WT"]).values,
    design="~ko_status", contrast=("ko_status", "recombined", "WT"),
    label="A: recombined vs WT")

# B -- recombined vs escaper, PAIRED within mouse
ko_mask = obs.ko_status.isin(["recombined", "escaper"]).values
paired_mice = [m for m in obs.loc[ko_mask, "mouse_id"].unique()
               if obs.loc[ko_mask & (obs.mouse_id == m).values, "ko_status"].nunique() == 2]
print(f"\nmice with BOTH a recombined and an escaper profile: {paired_mice}")
b_mask = ko_mask & obs.mouse_id.isin(paired_mice).values
results["B_recombined_vs_escaper_paired"] = run_contrast(
    pdata, b_mask,
    design="~mouse_id + ko_status" if len(paired_mice) > 1 else "~ko_status",
    contrast=("ko_status", "recombined", "escaper"),
    label="B: recombined vs escaper (paired)")

# C -- escaper vs WT, between animals
results["C_escaper_vs_WT"] = run_contrast(
    pdata, obs.ko_status.isin(["escaper", "WT"]).values,
    design="~ko_status", contrast=("ko_status", "escaper", "WT"),
    label="C: escaper vs WT (systemic)")

# D -- the naive comparison, for reference
# D must use EVERY beta cell, including `undetermined` ones -- it is meant to be
# the naive comparison you would have run without notebook 03b. Restricting it to
# classified cells would make it a different, biased contrast.
naive = adata[adata.obs.cell_type == "beta"].copy()
naive.obs["psb_naive"] = naive.obs["sample_id"].astype(str)
pnaive = dc.pp.pseudobulk(naive, sample_col="psb_naive", groups_col=None,
                          mode="sum", layer="counts")
pnaive.obs["genotype"] = pnaive.obs.index.map(
    naive.obs.drop_duplicates("sample_id").set_index("sample_id")["genotype"])
pnaive = pnaive[:, ~pnaive.var_names.isin(drop)].copy()
results["D_all_KO_vs_WT_naive"] = run_contrast(
    pnaive, np.ones(pnaive.n_obs, bool),
    design="~genotype", contrast=("genotype", "KO", "WT"),
    label="D: all KO beta vs WT (naive)")

de_all = pd.concat([r for r in results.values() if r is not None], ignore_index=True)
de_all.to_csv(cfg.TABLE_DIR / "05b_mosaic_de_all_contrasts.csv", index=False)

## 5b.3 Did mosaicism dilute the naive comparison as predicted?

If the classifier worked, contrast D's effect sizes should be approximately
`recombination_rate × contrast A`. A fitted slope close to the measured
recombination rate is strong independent evidence that the cell-level calls in
03b are correct — the two estimates come from entirely different information.

In [ ]:
A, D = results.get("A_recombined_vs_WT"), results.get("D_all_KO_vs_WT_naive")
if A is not None and D is not None:
    merged = A[["gene", "log2FoldChange", "padj"]].merge(
        D[["gene", "log2FoldChange"]], on="gene", suffixes=("_A", "_D"))
    sig = merged[merged.padj < cfg.FDR_ALPHA]

    fig, ax = plt.subplots(figsize=(5.2, 5))
    ax.scatter(merged.log2FoldChange_A, merged.log2FoldChange_D, s=4,
               c="#cccccc", rasterized=True)
    ax.scatter(sig.log2FoldChange_A, sig.log2FoldChange_D, s=10, c="#C44E52")
    lim = np.nanpercentile(np.abs(merged[["log2FoldChange_A", "log2FoldChange_D"]]), 99.5)
    ax.plot([-lim, lim], [-lim, lim], "k--", lw=1, label="y = x (no dilution)")
    ax.plot([-lim, lim], [-lim * MEAN_RATE, lim * MEAN_RATE], "-", c="#4C72B0", lw=1.5,
            label=f"y = {MEAN_RATE:.2f}x (predicted)")

    if len(sig) > 10:
        slope = np.polyfit(sig.log2FoldChange_A, sig.log2FoldChange_D, 1)[0]
        ax.plot([-lim, lim], [-lim * slope, lim * slope], "-", c="#8DA65B", lw=1.5,
                label=f"y = {slope:.2f}x (observed)")
        print(f"observed dilution slope {slope:.3f} vs recombination rate {MEAN_RATE:.3f}")
        if abs(slope - MEAN_RATE) < 0.15:
            print("  These agree. Independent confirmation that the 03b classification "
                  "is working — the two estimates share no information.")
        else:
            print("  These disagree. Either the classifier is mislabelling cells, or "
                  "there is a bystander effect making escapers non-neutral (check "
                  "contrast C).")

    ax.set_xlabel("log2FC — A (recombined vs WT)")
    ax.set_ylabel("log2FC — D (all KO vs WT)")
    ax.set_title("Mosaicism attenuates the naive contrast")
    ax.legend(fontsize=8, frameon=False); sns.despine(); fig.tight_layout()
    fig.savefig(cfg.FIG_DIR / "05b_dilution_check.png", dpi=300, bbox_inches="tight")

### How much is contrast B attenuated by imperfect classification?

Bola3 sits at roughly one count per cell, so per-cell calling is good but not
perfect — some recombined cells land in the escaper class and vice versa. That
pulls the two groups toward each other and shrinks the measured effect by
`(recombined_purity + escaper_purity - 1)`.

**This is conservative, not invalidating.** Misclassification can only shrink a
difference, never manufacture one, so a significant result from an impure
classification is still real — just an underestimate. Combined with survivor
bias, which points the same way, treat every effect size in contrast B as a
floor rather than a point estimate.

In [ ]:
if {"expected_rec_purity", "expected_esc_purity"} <= set(rates.columns):
    att = attenuation_from_impurity(rates.expected_rec_purity.mean(),
                                    rates.expected_esc_purity.mean())
    for k, v in att.items():
        print(f"  {k}: {v}")
    if B is not None:
        B_corrected = B.copy()
        B_corrected["log2FoldChange_purity_corrected"] = (
            B_corrected["log2FoldChange"] / att["observed_over_true"])
        B_corrected.to_csv(cfg.TABLE_DIR / "05b_contrastB_purity_corrected.csv")
        print("\n  wrote purity-corrected log2FCs for contrast B.")
        print("  Report the RAW values as your result and the corrected ones as an")
        print("  estimate of true magnitude -- never the other way round.")

## 5b.4 Comparing the contrasts

In [ ]:
summary = pd.DataFrame([
    {"contrast": label,
     "n_tested": len(r),
     "n_sig": int((r.padj < cfg.FDR_ALPHA).sum()),
     "n_up": int(((r.padj < cfg.FDR_ALPHA) & (r.log2FoldChange > 0)).sum()),
     "n_down": int(((r.padj < cfg.FDR_ALPHA) & (r.log2FoldChange < 0)).sum()),
     "median_abs_lfc_sig": round(r.loc[r.padj < cfg.FDR_ALPHA, "log2FoldChange"].abs().median(), 3)}
    for label, r in results.items() if r is not None
])
display(summary)
summary.to_csv(cfg.TABLE_DIR / "05b_contrast_summary.csv", index=False)

fig, axes = plt.subplots(1, len([r for r in results.values() if r is not None]),
                         figsize=(4.6 * len(summary), 4.4))
key_genes = ["Bola3", "Lias", "Sdhb", "Nfu1", "Atf4", "Ddit3", "Fgf21", "Gdf15",
             "Ucn3", "Mafa", "Slc2a2", "G6pc2", "Aldh1a3", "Ldha", "Slc16a1"]
for ax, (label, res) in zip(np.atleast_1d(axes), [(k, v) for k, v in results.items() if v is not None]):
    d = res.dropna(subset=["padj"]).copy()
    d["nlp"] = -np.log10(d.padj.clip(lower=1e-300))
    sig = d.padj < cfg.FDR_ALPHA
    ax.scatter(d.loc[~sig, "log2FoldChange"], d.loc[~sig, "nlp"], s=4, c="#cccccc", rasterized=True)
    ax.scatter(d.loc[sig, "log2FoldChange"], d.loc[sig, "nlp"], s=8,
               c=np.where(d.loc[sig, "log2FoldChange"] > 0, "#C44E52", "#4C72B0"))
    for g in key_genes:
        if g in d.index:
            r = d.loc[g]
            ax.annotate(g, (r.log2FoldChange, r.nlp), fontsize=7,
                        xytext=(3, 3), textcoords="offset points")
    ax.axhline(-np.log10(cfg.FDR_ALPHA), ls="--", c="k", lw=0.7)
    ax.set_title(f"{label}\n{sig.sum()} sig", fontsize=9)
    ax.set_xlabel("log2FC")
sns.despine(fig); fig.tight_layout()
fig.savefig(cfg.FIG_DIR / "05b_volcanoes_four_contrasts.png", dpi=300, bbox_inches="tight")

## 5b.5 Separating cell-autonomous from systemic effects

Plotting contrast B against contrast C partitions every gene:

- **B large, C ≈ 0** → cell-autonomous. Requires Bola3 loss *in that cell*.
- **B ≈ 0, C large** → systemic. Present in unrecombined cells of KO animals;
  attributable to the glucose intolerance, not to Bola3 in that cell.
- **Both large, same sign** → the cell-autonomous lesion and the systemic
  environment push the same way. Common for stress-response genes, and the
  hardest category to interpret.

In [ ]:
B, C = results.get("B_recombined_vs_escaper_paired"), results.get("C_escaper_vs_WT")
if B is not None and C is not None:
    m = B[["gene", "log2FoldChange", "padj"]].merge(
        C[["gene", "log2FoldChange", "padj"]], on="gene", suffixes=("_B", "_C"))
    m["class"] = np.select(
        [(m.padj_B < cfg.FDR_ALPHA) & (m.padj_C >= cfg.FDR_ALPHA),
         (m.padj_B >= cfg.FDR_ALPHA) & (m.padj_C < cfg.FDR_ALPHA),
         (m.padj_B < cfg.FDR_ALPHA) & (m.padj_C < cfg.FDR_ALPHA)],
        ["cell-autonomous", "systemic", "both"], default="neither")

    fig, ax = plt.subplots(figsize=(6, 5.5))
    colours = {"cell-autonomous": "#C44E52", "systemic": "#4C72B0",
               "both": "#9B59B6", "neither": "#dddddd"}
    for cls, colour in colours.items():
        s = m[m["class"] == cls]
        ax.scatter(s.log2FoldChange_B, s.log2FoldChange_C, s=10 if cls != "neither" else 4,
                   c=colour, label=f"{cls} (n={len(s)})", rasterized=(cls == "neither"))
    for g in key_genes:
        r = m[m.gene == g]
        if len(r):
            ax.annotate(g, (r.log2FoldChange_B.iloc[0], r.log2FoldChange_C.iloc[0]),
                        fontsize=7, xytext=(3, 3), textcoords="offset points")
    ax.axhline(0, c="k", lw=0.6); ax.axvline(0, c="k", lw=0.6)
    ax.set_xlabel("log2FC — B: cell-autonomous (recombined vs escaper)")
    ax.set_ylabel("log2FC — C: systemic (escaper vs WT)")
    ax.legend(fontsize=8, frameon=False)
    sns.despine(); fig.tight_layout()
    fig.savefig(cfg.FIG_DIR / "05b_autonomous_vs_systemic.png", dpi=300, bbox_inches="tight")

    display(m["class"].value_counts())
    m.to_csv(cfg.TABLE_DIR / "05b_autonomous_vs_systemic.csv", index=False)

## 5b.6 Your specific prediction: insulin secretion coupling

Your islets show impaired GSIS **in vitro** — isolated from the animal, away
from its circulation, they still fail to secrete. That localises the defect to
the beta cell itself, which makes a sharp prediction:

> The insulin secretion coupling machinery should be disrupted in **contrast B**
> (cell-autonomous) and not merely in contrast C (systemic).

High fasting insulin with low stimulated insulin and a low stimulation index is
the signature of a beta cell that cannot translate glucose into an ATP/ADP rise.
Genes to watch: `Gck`, `Slc2a2` (glucose sensing), the OXPHOS complexes (ATP
generation), `Abcc8`/`Kcnj11` (K-ATP), `Cacna1d` (calcium entry), and the
anaplerotic enzymes `Pcx`, `Mdh1`, `Me1`.

In [ ]:
from bola3_gene_sets import DIRECTIONAL_HYPOTHESES, GENE_SETS, as_decoupler_net, filter_sets
from statsmodels.stats.multitest import multipletests

sets = filter_sets(GENE_SETS, pdata.var_names, min_genes=5, verbose=False)
net = as_decoupler_net(sets)

rows = []
for label, res in results.items():
    if res is None:
        continue
    ranks = res.dropna(subset=["stat"]).set_index("gene")[["stat"]].T
    ranks.index = [label]
    try:
        nes, pval = dc.mt.gsea(data=ranks, net=net, tmin=5)
        rows.append(pd.DataFrame({"contrast": label, "gene_set": nes.columns,
                                  "nes": nes.iloc[0].values, "p": pval.iloc[0].values}))
    except Exception as e:
        print(f"  {label}: {e}")

gsea = pd.concat(rows, ignore_index=True)
gsea["FDR"] = np.nan
for c, idx in gsea.groupby("contrast").groups.items():
    gsea.loc[idx, "FDR"] = multipletests(gsea.loc[idx, "p"], method="fdr_bh")[1]
gsea["predicted"] = gsea.gene_set.map(DIRECTIONAL_HYPOTHESES)
gsea.to_csv(cfg.TABLE_DIR / "05b_gsea_by_contrast.csv", index=False)

order = [k for k in DIRECTIONAL_HYPOTHESES if k in set(gsea.gene_set)]
pivot = gsea.pivot_table(index="gene_set", columns="contrast", values="nes").reindex(order)
sigmat = (gsea.pivot_table(index="gene_set", columns="contrast", values="FDR")
          .reindex(order) < 0.05)
annot = pivot.round(2).astype(str).where(~sigmat, pivot.round(2).astype(str) + "*")

fig, ax = plt.subplots(figsize=(2.2 * pivot.shape[1] + 4, 6.5))
sns.heatmap(pivot, cmap="RdBu_r", center=0, annot=annot, fmt="", ax=ax,
            linewidths=0.5, cbar_kws={"label": "GSEA NES"})
ax.set_title("Mechanism scorecard across the four contrasts  (* FDR<0.05)\n"
             "column B is the cell-autonomous effect", fontsize=10)
ax.set_xlabel(""); ax.set_ylabel("")
fig.tight_layout()
fig.savefig(cfg.FIG_DIR / "05b_scorecard_by_contrast.png", dpi=300, bbox_inches="tight")

In [ ]:
secretion = [g for g in GENE_SETS["Insulin_secretion_coupling"] if B is not None and g in B.index]
if B is not None and secretion:
    tbl = B.loc[secretion, ["log2FoldChange", "pvalue", "padj"]].sort_values("log2FoldChange")
    tbl["significant"] = tbl.padj < cfg.FDR_ALPHA
    display(tbl.round(4))

    fig, ax = plt.subplots(figsize=(6, 0.28 * len(tbl) + 1.5))
    ax.barh(tbl.index, tbl.log2FoldChange,
            color=["#C44E52" if s else "#cccccc" for s in tbl.significant])
    ax.axvline(0, c="k", lw=1)
    ax.set_xlabel("log2FC, recombined vs escaper (cell-autonomous)")
    ax.set_title("Insulin secretion coupling machinery\n"
                 "your in vitro GSIS defect predicts these fall", fontsize=10)
    sns.despine(); fig.tight_layout()
    fig.savefig(cfg.FIG_DIR / "05b_secretion_coupling.png", dpi=300, bbox_inches="tight")

---
## Reporting this

Lead with **contrast B**. The sentence you are aiming for is something like:

> "Within individual Bola3^fl/fl; Pdx1-CreER animals, beta cells in which
> recombination had occurred were compared against escaper beta cells from the
> same islets. Recombined cells showed reduced expression of [...], while
> escaper cells were indistinguishable from tamoxifen-treated Cre-negative
> controls, indicating a cell-autonomous requirement for Bola3."

That construction pre-empts the two obvious objections — mouse-to-mouse
variation, and confounding by the animals' glucose intolerance — because the
design excludes both rather than arguing them away.

Also state, in the methods or the discussion:

- per-mouse recombination rates, and the method used to estimate them
- the posterior cutoff, and that `undetermined` cells were excluded
- **survivor bias**: at an 8-month chase, the most severely affected recombined
  cells may already have died, so reported effect sizes are a lower bound
- that preferential recombination of mature beta cells cannot be excluded
  without a lineage reporter (§3b.5 bounds how much it could matter)